# Notebook 01: Data Loading and Preprocessing

This notebook covers the first stage of the pipeline: reading shoulder X-ray images from disk,
extracting clinical metadata embedded in the file, and producing a normalized grayscale array
ready for inference.

**What you will learn:**
- The structure of the DICOM format and why radiology uses it
- How to extract clinical metadata (laterality, patient age, sex) from DICOM tags
- The full normalization chain: VOI LUT windowing, photometric inversion, min-max scaling
- How to handle standard image formats (JPG, PNG) as a fallback
- A pixel-intensity heuristic to infer shoulder laterality when the DICOM tag is missing

**Files used:**
- `../Frontend/Images/I_AP (1).dcm` — a real AP (anteroposterior) shoulder DICOM
- `../Frontend/Images/shoulder_x-ray.jpg` — a standard JPG for comparison

**Source code reference:** `Backend/image_utils.py`, function `procesar_imagen_base()`

## 1. Clinical Background

The rotator cuff is a group of four muscles and their tendons that surround the shoulder joint
(supraspinatus, infraspinatus, subscapularis, teres minor). Calcific tendinopathy occurs when
calcium hydroxyapatite crystals deposit inside the tendon tissue, causing pain, limited range of
motion, and in some cases acute inflammatory crises.

Plain X-rays are the standard first-line imaging study for detecting and classifying these deposits.
Two standard projections are used:
- **AP (anteroposterior):** the X-ray beam passes from front to back. The patient may hold the arm
  in internal rotation (RI) or external rotation (RE) to expose different parts of the tendon.
- **Axial / Y-view:** less common in this dataset.

Calcifications are classified by morphology (Gartner classification, Types I-III) based on their
density, contour, and homogeneity on X-ray:
- **Type I:** dense, well-defined, homogeneous
- **Type II:** intermediate characteristics
- **Type III:** translucent, poorly defined, heterogeneous

This project automates the detection and classification of these deposits using deep learning and
classical radiomics.

## 2. Imports

In [ ]:
import numpy as np
import cv2
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import matplotlib.pyplot as plt
import re

# Paths relative to this notebook
DICOM_PATH = "../Frontend/Images/I_AP (1).dcm"
JPG_PATH   = "../Frontend/Images/shoulder_x-ray.jpg"

## 3. The DICOM Format

DICOM (Digital Imaging and Communications in Medicine) is the international standard for storing
and transmitting medical images. Unlike JPEG or PNG, a DICOM file bundles the pixel data with a
structured header containing hundreds of standardized tags that describe the acquisition, the
equipment, and the patient.

Each tag is identified by a pair of hexadecimal numbers `(group, element)`. Relevant tags for
this project:

| Tag | Name | Example value |
|-----|------|---------------|
| (0020,0060) | Laterality | `R` or `L` |
| (0020,0062) | ImageLaterality | `R` or `L` |
| (0010,1010) | PatientAge | `065Y` |
| (0010,0040) | PatientSex | `M` or `F` |
| (0028,1050) | WindowCenter | windowing parameter |
| (0028,1051) | WindowWidth  | windowing parameter |
| (0028,0004) | PhotometricInterpretation | `MONOCHROME2` or `MONOCHROME1` |

In [ ]:
ds = pydicom.dcmread(DICOM_PATH)

# Print a selection of relevant tags
tags_of_interest = {
    "Modality":                  getattr(ds, "Modality", "N/A"),
    "ImageLaterality":           getattr(ds, "ImageLaterality", getattr(ds, "Laterality", "N/A")),
    "PatientAge":                getattr(ds, "PatientAge", "N/A"),
    "PatientSex":                getattr(ds, "PatientSex", "N/A"),
    "PhotometricInterpretation": getattr(ds, "PhotometricInterpretation", "N/A"),
    "WindowCenter":              getattr(ds, "WindowCenter", "N/A"),
    "WindowWidth":               getattr(ds, "WindowWidth", "N/A"),
    "Rows":                      getattr(ds, "Rows", "N/A"),
    "Columns":                   getattr(ds, "Columns", "N/A"),
    "PixelSpacing":              getattr(ds, "PixelSpacing", "N/A"),
}

for key, val in tags_of_interest.items():
    print(f"{key:30s}: {val}")

### 3.1 Extracting Structured Metadata

Before reading the pixel data, the pipeline extracts three pieces of clinical information
that are later used by the hybrid model:

1. **Laterality** (`R` / `L`) — tells the model which side of the body is being imaged.
   Left shoulders are horizontally flipped before classification so the model always
   sees the anatomical structures in the same orientation.

2. **Patient age** — stored as a string like `065Y`. The numeric part is extracted with a
   regex and used as a continuous feature in the hybrid ML model.

3. **Patient sex** — encoded as `M` (male = 1) or `F` (female = 0), also a hybrid model feature.
   Calcific tendinopathy has a higher prevalence in women aged 40-60.

In [ ]:
# --- Laterality ---
tag_lat = getattr(ds, "ImageLaterality", getattr(ds, "Laterality", None))
laterality = {"R": "Right", "L": "Left"}.get(tag_lat, "Unknown") if tag_lat else "Unknown"

# --- Age ---
age_raw = getattr(ds, "PatientAge", None)
age = None
if age_raw:
    digits = re.sub(r"[^0-9]", "", str(age_raw))
    age = int(digits) if digits else None

# --- Sex ---
sex_str = getattr(ds, "PatientSex", None)
sex = {"M": 1, "F": 0}.get(sex_str, None)

print(f"Laterality : {laterality}")
print(f"Patient age: {age}")
print(f"Patient sex: {sex_str} -> encoded as {sex}")

## 4. Reading and Normalizing the Pixel Data

DICOM pixel values are raw sensor measurements stored in 12 or 16 bits. They cannot be displayed
directly because:

1. **The useful range of values is narrow.** Bone, soft tissue, and air each absorb X-rays
   differently. Radiologists use a *window* (center + width) to map the relevant range to
   the 0-255 display range. This is called VOI LUT (Value of Interest Look-Up Table).

2. **Photometric convention varies.** `MONOCHROME2` encodes bone as high values (bright).
   `MONOCHROME1` inverts this convention (bone is dark). The image must be inverted to
   produce a consistent representation.

3. **Absolute scale differs between scanners.** A min-max normalization to `[0, 255]` ensures
   all images occupy the same dynamic range regardless of the acquisition device.

In [ ]:
# Step 1: Apply VOI LUT windowing (uses WindowCenter / WindowWidth tags)
if "WindowWidth" in ds and "WindowCenter" in ds:
    img_array = apply_voi_lut(ds.pixel_array, ds)
    print("VOI LUT applied using embedded window parameters.")
else:
    img_array = ds.pixel_array.astype(float)
    print("No windowing tags found. Using raw pixel values.")

print(f"Raw pixel range after VOI LUT: [{img_array.min():.0f}, {img_array.max():.0f}]")
print(f"Array dtype: {img_array.dtype}, shape: {img_array.shape}")

In [ ]:
# Step 2: Invert MONOCHROME1 images so that bone always appears bright
photometric = getattr(ds, "PhotometricInterpretation", "MONOCHROME2")
if photometric == "MONOCHROME1":
    img_array = np.amax(img_array) - img_array
    print("Photometric inversion applied (MONOCHROME1 -> MONOCHROME2).")
else:
    print(f"No inversion needed (PhotometricInterpretation = {photometric}).")

In [ ]:
# Step 3: Min-max normalization to uint8 [0, 255]
img_array = img_array - np.min(img_array)
if np.max(img_array) != 0:
    img_array = img_array / np.max(img_array)

img_uint8 = (img_array * 255).astype(np.uint8)

print(f"Final array dtype : {img_uint8.dtype}")
print(f"Final value range : [{img_uint8.min()}, {img_uint8.max()}]")
print(f"Final shape       : {img_uint8.shape}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img_uint8, cmap="gray")
plt.title(f"Processed DICOM — {laterality} shoulder — Age {age}")
plt.axis("off")
plt.tight_layout()
plt.show()

## 5. Loading a Standard Image (JPG / PNG)

When the input is a JPG or PNG (for example, a scanned film or a screenshot of a workstation),
there are no DICOM metadata tags available. The processing is simpler:

1. Read with OpenCV in grayscale mode.
2. Apply the same min-max normalization to ensure consistency with DICOM-derived images.

The absence of metadata means laterality must be inferred from pixel content, and clinical
variables (age, sex) fall back to population-level defaults.

In [ ]:
img_jpg_raw = cv2.imread(JPG_PATH, cv2.IMREAD_GRAYSCALE)
assert img_jpg_raw is not None, f"Could not read file: {JPG_PATH}"

img_jpg = img_jpg_raw.astype(np.float32)
img_jpg = (img_jpg - img_jpg.min()) / (img_jpg.max() - img_jpg.min()) * 255
img_jpg = img_jpg.astype(np.uint8)

print(f"JPG shape: {img_jpg.shape}, dtype: {img_jpg.dtype}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_uint8, cmap="gray")
axes[0].set_title(f"DICOM  ({img_uint8.shape[1]} x {img_uint8.shape[0]} px)")
axes[0].axis("off")

axes[1].imshow(img_jpg, cmap="gray")
axes[1].set_title(f"JPG  ({img_jpg.shape[1]} x {img_jpg.shape[0]} px)")
axes[1].axis("off")

plt.suptitle("Comparison: DICOM vs JPG after normalization", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Laterality Heuristic

When the DICOM tag is absent or when the image is a JPG, the pipeline uses a simple
pixel-intensity heuristic to decide which side of the body is shown.

**Assumption:** in an AP shoulder X-ray, the side containing the humeral head (bright bone)
tends to have a higher mean pixel intensity than the opposite side.

The image is split at the horizontal midpoint. The half with the higher mean intensity is
assumed to be the side that contains the shoulder joint.

- If the **right** half is brighter -> the shoulder is on the **right**
- If the **left** half is brighter -> the shoulder is on the **left**

This heuristic works well for isolated shoulder films but may fail on full-chest radiographs
or images with very similar bilateral brightness.

In [ ]:
def infer_laterality(img: np.ndarray) -> str:
    h, w = img.shape
    avg_left  = np.mean(img[:, : w // 2])
    avg_right = np.mean(img[:, w // 2 :])
    return "Right" if avg_right > avg_left else "Left"

# Apply to both images
lat_dicom = infer_laterality(img_uint8)
lat_jpg   = infer_laterality(img_jpg)

print(f"DICOM tag laterality    : {laterality}")
print(f"DICOM inferred laterality: {lat_dicom}")
print(f"JPG inferred laterality : {lat_jpg}")

# Visualize the split
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.imshow(img_uint8, cmap="gray")
ax.axvline(x=img_uint8.shape[1] // 2, color="red", linewidth=1.5, linestyle="--", label="midpoint")
ax.set_title(f"Laterality heuristic (detected: {lat_dicom})")
ax.legend(loc="upper right")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Summary

The preprocessing function `procesar_imagen_base()` in `Backend/image_utils.py` encapsulates
all the steps above into a single call that accepts any file stream (DICOM or standard image)
and returns a normalized `uint8` grayscale array plus extracted metadata.

The full preprocessing chain is:

```
Input file (DICOM / JPG / PNG)
        |
        v
Read pixel data
        |
        v
Apply VOI LUT  (DICOM only, if window tags present)
        |
        v
Invert MONOCHROME1  (DICOM only, if needed)
        |
        v
Min-max normalize -> uint8 [0, 255]
        |
        v
Extract laterality  (DICOM tag > heuristic fallback)
        |
        v
Extract age / sex   (DICOM tags only, or default values)
        |
        v
Output: (img_uint8, laterality, {age, sex})
```

The resulting array is the input to Notebook 02, which uses a U-Net model to crop the image
to the shoulder region of interest before classification.